In [1]:
!pip install gensim nltk
import nltk
nltk.download('punkt', quiet=True)


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


True

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Dataset/cleaned_data.csv')
docs = df['clean_text'].astype(str).tolist()

print(f"Total Data: {len(docs)}")

Total Data: 20000


In [3]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(docs)

vocab = vectorizer.get_feature_names_out()
print("Vocabulary Size:", len(vocab))
print("Sample Vocab (10 pertama):", vocab[:10])
print("Matrix Shape:", X_bow.shape)

Vocabulary Size: 869
Sample Vocab (10 pertama): ['ability' 'able' 'accept' 'according' 'account' 'across' 'act' 'action'
 'activity' 'actually']
Matrix Shape: (20000, 869)


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(norm=None, use_idf=True, smooth_idf=False)
X_tfidf = tfidf.fit_transform(docs)

vocab = tfidf.get_feature_names_out()
idf = tfidf.idf_

print("Vocabulary (10 pertama):", vocab[:10])

print("\nIDF Values (10 pertama):")
for word, val in zip(vocab[:10], idf[:10]):
    print(f"{word}: {round(val, 3)}")

print("\nTF-IDF Matrix Shape:", X_tfidf.shape)
tfidf_matrix = X_tfidf.toarray()
print("Sample Matrix (3 baris x 5 kolom):")
print(tfidf_matrix[:3, :5])

Vocabulary (10 pertama): ['ability' 'able' 'accept' 'according' 'account' 'across' 'act' 'action'
 'activity' 'actually']

IDF Values (10 pertama):
ability: 2.452
able: 2.48
accept: 2.482
according: 2.496
account: 2.468
across: 2.473
act: 2.464
action: 2.502
activity: 2.49
actually: 2.465

TF-IDF Matrix Shape: (20000, 869)
Sample Matrix (3 baris x 5 kolom):
[[0.         0.         0.         0.         0.        ]
 [2.45179334 0.         2.48192459 0.         0.        ]
 [2.45179334 0.         0.         0.         0.        ]]


In [5]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)  # fallback untuk kompatibilitas

True

In [6]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

tokenized_docs = [word_tokenize(doc) for doc in docs]

w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    epochs=10
)

def doc_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

X_w2v = np.array([doc_vector(doc, w2v_model) for doc in tokenized_docs])

print("Word2Vec Shape:", X_w2v.shape)

Word2Vec Shape: (20000, 100)


In [7]:
np.save('X_tfidf.npy', tfidf_matrix)
np.save('X_w2v.npy', X_w2v)

y = df['label'].map({'real': 0, 'fake': 1}).values
np.save('y_labels.npy', y)

print("File berhasil disimpan: X_tfidf.npy, X_w2v.npy, y_labels.npy")

File berhasil disimpan: X_tfidf.npy, X_w2v.npy, y_labels.npy


# MODELING

In [11]:
# Cell Code 1: Load Data & Train-Test Split
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load data
print("Loading data...")
X_tfidf = np.load('X_tfidf.npy')
X_w2v = np.load('X_w2v.npy')
y = np.load('y_labels.npy')

# 2. Split data (80% Training, 20% Testing)
# Split untuk TF-IDF
X_train_tf, X_test_tf, y_train_tf, y_test_tf = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# Split untuk Word2Vec
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data split selesai!")
print(f"Shape Training TF-IDF: {X_train_tf.shape}")
print(f"Shape Testing TF-IDF: {X_test_tf.shape}")
print(f"Shape Training Word2Vec: {X_train_w2v.shape}")
print(f"Shape Testing Word2Vec: {X_test_w2v.shape}")

Loading data...
Data split selesai!
Shape Training TF-IDF: (16000, 869)
Shape Testing TF-IDF: (4000, 869)
Shape Training Word2Vec: (16000, 100)
Shape Testing Word2Vec: (4000, 100)


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

# --- FUNGSI EVALUASI ---
def evaluate_model(model_name, feature_name, y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"--- Selesai melatih: {model_name} ({feature_name}) ---")
    return {
        'Model': model_name,
        'Feature Representation': feature_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }
experiment_results = []

In [13]:
#LOGISTIC REGRESSION
log_reg_model = LogisticRegression(max_iter=1000)

In [ ]:
# Logistic Regression + TF-IDF
log_reg_model.fit(X_train_tf, y_train_tf)
y_pred_lr_tf = log_reg_model.predict(X_test_tf)

res_lr_tf = evaluate_model("Logistic Regression", "TF-IDF", y_test_tf, y_pred_lr_tf)
experiment_results.append(res_lr_tf)

--- Selesai melatih: Logistic Regression (TF-IDF) ---


In [ ]:
# Logistic Regression + Word2Vec
log_reg_model.fit(X_train_w2v, y_train_w2v)
y_pred_lr_w2v = log_reg_model.predict(X_test_w2v)

res_lr_w2v = evaluate_model("Logistic Regression", "Word2Vec", y_test_w2v, y_pred_lr_w2v)
experiment_results.append(res_lr_w2v)

--- Selesai melatih: Logistic Regression (Word2Vec) ---


In [ ]:
# Naive Bayes + TF-IDF
nb_model = GaussianNB()

nb_model.fit(X_train_tf, y_train_tf)
y_pred_nb_tf = nb_model.predict(X_test_tf)

res_nb_tf = evaluate_model("Naive Bayes", "TF-IDF", y_test_tf, y_pred_nb_tf)
experiment_results.append(res_nb_tf)

--- Selesai melatih: Naive Bayes (TF-IDF) ---


In [ ]:
# Naive Bayes + Word2Vec
nb_model.fit(X_train_w2v, y_train_w2v)
y_pred_nb_w2v = nb_model.predict(X_test_w2v)

res_nb_w2v = evaluate_model("Naive Bayes", "Word2Vec", y_test_w2v, y_pred_nb_w2v)
experiment_results.append(res_nb_w2v)

--- Selesai melatih: Naive Bayes (Word2Vec) ---


COMPARISON

In [21]:
# Menampilkan Komparasi
results_df = pd.DataFrame(experiment_results)

print("\nHASIL EKSPERIMEN & KOMPARASI")
display(results_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True))


HASIL EKSPERIMEN & KOMPARASI


,Model,Feature Representation,Accuracy,Precision,Recall,F1-Score
0,Naive Bayes,Word2Vec,0.50900,0.511239,0.531576,0.521209
1,Logistic Regression,TF-IDF,0.50800,0.510432,0.523123,0.516699
2,Logistic Regression,Word2Vec,0.50275,0.502750,1.000000,0.669107
3,Naive Bayes,TF-IDF,0.50275,0.505366,0.515167,0.510219
